# 📘 캡슐화 (Encapsulation)

**캡슐화**는 객체의 내부 상태를 보호하고, 외부에서 제어된 방식으로만 접근하도록 하는 개념입니다.
파이썬은 `_`(보호)와 `__`(비공개) 네이밍 컨벤션을 사용합니다.

**학습 목표:**
- public, protected, private 속성
- getter/setter와 @property
- 읽기 전용 속성과 계산된 속성
- @property.deleter와 __slots__

## 1. 접근 제어 — public, protected, private

파이썬의 접근 제어는 네이밍 컨벤션에 의존합니다.
`_`(밑줄 1개)은 보호, `__`(밑줄 2개)은 비공개를 의미합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  파이썬 접근 제어 컨벤션                   │
# │  self.public     → 공개 (어디서나 접근)      │
# │  self._protected → 보호 (관례상 내부만)      │
# │  self.__private  → 비공개 (name mangling)    │
# └─────────────────────────────────────────┘

class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner           # 공개 속성
        self._bank_code = "BANK001"  # 보호 속성 (관례)
        self.__balance = balance     # 비공개 속성 (name mangling)

    def get_balance(self):
        return self.__balance

    def set_balance(self, amount):
        if amount < 0:
            raise ValueError("잔액은 음수가 될 수 없습니다")
        self.__balance = amount

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("입금액은 양수여야 합니다")
        self.__balance += amount
        return self.__balance

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("출금액은 양수여야 합니다")
        if amount > self.__balance:
            raise ValueError("잔액이 부족합니다")
        self.__balance -= amount
        return self.__balance

account = BankAccount("김파이", 1000)

print(f"소유자: {account.owner}")           # 공개 속성
print(f"은행 코드: {account._bank_code}")   # 보호 속성
print(f"잔액: {account.get_balance()}")     # 비공개 → getter로

In [ ]:
# name mangling — 비공개 속성의 실제 이름
# __balance → _BankAccount__balance 로 변환됨

# 직접 접근 시도 (에러)
# print(account.__balance)  # AttributeError!

# name mangling을 통한 우회 (가능하지만 권장하지 않음)
print(f"name mangling: {account._BankAccount__balance}")

# 입금과 출금
account.deposit(500)
print(f"입금 후 잔액: {account.get_balance()}")

account.withdraw(200)
print(f"출금 후 잔액: {account.get_balance()}")

## 2. @property — 파이썬스러운 getter/setter

`@property`를 사용하면 메서드를 속성처럼 접근할 수 있습니다.
`@속성명.setter`로 설정 로직을, `@속성명.deleter`로 삭제 로직을 추가할 수 있습니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  @property 사용법                          │
# │  @property         → getter (읽기)          │
# │  @속성명.setter     → setter (쓰기)          │
# │  @속성명.deleter    → deleter (삭제)         │
# └─────────────────────────────────────────┘

class Temperature:
    def __init__(self, celsius=0):
        self.celsius = celsius  # setter를 통해 검증됨

    @property
    def celsius(self):
        """섭씨 온도 (getter)"""
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        """섭씨 온도 (setter — 검증 포함)"""
        if value < -273.15:
            raise ValueError("절대영도 이하의 온도는 불가합니다")
        self._celsius = value

    @property
    def fahrenheit(self):
        """화씨 온도 (읽기 전용 계산 속성)"""
        return self._celsius * 9 / 5 + 32

    @property
    def kelvin(self):
        """켈빈 온도 (읽기 전용 계산 속성)"""
        return self._celsius + 273.15

t = Temperature(25)
print(f"섭씨: {t.celsius}°C")
print(f"화씨: {t.fahrenheit}°F")
print(f"켈빈: {t.kelvin}K")

# setter를 통한 값 변경 (검증 포함)
t.celsius = 100
print(f"변경 후: {t.celsius}°C")

try:
    t.celsius = -300
except ValueError as e:
    print(f"에러: {e}")

In [ ]:
# ┌─────────────────────────────────────────┐
# │  @property.deleter와 __slots__              │
# │  deleter → 속성 삭제 시 동작 정의            │
# │  __slots__ → 허용 속성 제한 (메모리 절약)    │
# └─────────────────────────────────────────┘

class Person:
    __slots__ = ['_name', '_age']  # 허용 속성 제한

    def __init__(self, name, age):
        self.name = name
        self.age = age

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        if not value:
            raise ValueError("이름은 비워둘 수 없습니다")
        self._name = value

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if value < 0:
            raise ValueError("나이는 음수가 될 수 없습니다")
        self._age = value

p = Person("김파이", 25)
print(f"이름: {p.name}, 나이: {p.age}")

p.name = "이코딩"
p.age = 30
print(f"변경: {p.name}, {p.age}")

# __slots__가 없는 속성 추가 불가
# p.email = "test@test.com"  # AttributeError!

## 🎯 연습 문제

1. `BankAccount` 클래스에 `@property`를 사용해 `balance` 속성을 음수 방지 setter로 만드세요.
2. `Temperature` 클래스에 `fahrenheit` 속성에 setter를 추가해 화씨로 온도를 설정할 수 있게 하세요.
3. `__slots__ = ['_x', '_y']`를 가진 `Point` 클래스를 작성하고, `x`, `y`를 `@property`로 제어하세요.
4. `@property.deleter`를 사용해 속성 삭제 시 `"속성명이 삭제되었습니다"`를 출력하는 예제를 작성하세요.